<!--BOOK_INFORMATION-->
<img align="left" style="padding-right:10px;" src="figures/PDSH-cover-small.png">
*This notebook contains an excerpt from the [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) by Jake VanderPlas; the content is available [on GitHub](https://github.com/jakevdp/PythonDataScienceHandbook).*

*The text is released under the [CC-BY-NC-ND license](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), and code is released under the [MIT license](https://opensource.org/licenses/MIT). If you find this content useful, please consider supporting the work by [buying the book](http://shop.oreilly.com/product/0636920034919.do)!*

<!--NAVIGATION-->
< [Introducing Pandas Objects](03.01-Introducing-Pandas-Objects.ipynb) | [Contents](Index.ipynb) | [Operating on Data in Pandas](03.03-Operations-in-Pandas.ipynb) >

# 3.3 Data Indexing and Selection / 数据的取值与选择

In [Chapter 2](02.00-Introduction-to-NumPy.ipynb), we looked in detail at methods and tools to access, set, and modify values in NumPy arrays.
These included indexing (e.g., ``arr[2, 1]``), slicing (e.g., ``arr[:, 1:5]``), masking (e.g., ``arr[arr > 0]``), fancy indexing (e.g., ``arr[0, [1, 5]]``), and combinations thereof (e.g., ``arr[:, [1, 5]]``).
Here we'll look at similar means of accessing and modifying values in Pandas ``Series`` and ``DataFrame`` objects.
If you have used the NumPy patterns, the corresponding patterns in Pandas will feel very familiar, though there are a few quirks to be aware of.

🐍 取值操作，切片操作，掩码操作，花哨的索引操作，以及组合操作

We'll start with the simple case of the one-dimensional ``Series`` object, and then move on to the more complicated two-dimesnional ``DataFrame`` object.

## 3.3.1 Data Selection in Series / Series 数据选择方法

As we saw in the previous section, a ``Series`` object acts in many ways like a one-dimensional NumPy array, and in many ways like a standard Python dictionary.
If we keep these two overlapping analogies in mind, it will help us to understand the patterns of data indexing and selection in these arrays.

### 1. Series as dictionary / 将 Series 看作字典

Like a dictionary, the ``Series`` object provides a mapping from a collection of keys to a collection of values:

In [88]:
# 🐍 和字典一样，Series 对象提供了键值对的映射：

import pandas as pd
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', '2', '3', '4'])
data

a    0.25
2    0.50
3    0.75
4    1.00
dtype: float64

In [4]:
data['4']

1.0

We can also use dictionary-like Python expressions and methods to examine the keys/indices and values:

In [89]:
'4' in data

True

In [91]:
data.keys()

Index(['a', '2', '3', '4'], dtype='object')

In [90]:
dict(data.items())

{'a': 0.25, '2': 0.5, '3': 0.75, '4': 1.0}

In [93]:
data.values

array([0.25, 0.5 , 0.75, 1.  ])

In [94]:
data.keys

<bound method Series.keys of a    0.25
2    0.50
3    0.75
4    1.00
dtype: float64>

In [95]:
data.items

<bound method Series.items of a    0.25
2    0.50
3    0.75
4    1.00
dtype: float64>

``Series`` objects can even be modified with a dictionary-like syntax.
Just as you can extend a dictionary by assigning to a new key, you can extend a ``Series`` by assigning to a new index value:

In [96]:
# 🐍 就像可以通过增加新的键扩展字典一样，也可以通过增加新的索引值扩展Series

data['y'] = 'new'
data['e'] = 'e'
data

a    0.25
2     0.5
3    0.75
4     1.0
y     new
e       e
dtype: object

This easy mutability of the objects is a convenient feature: under the hood, Pandas is making decisions about memory layout and data copying that might need to take place; the user generally does not need to worry about these issues.

### 2. Series as one-dimensional array / 将 Series 看作一维数组

A ``Series`` builds on this dictionary-like interface and provides array-style item selection via the same basic mechanisms as NumPy arrays – that is, *slices*, *masking*, and *fancy indexing*.
Examples of these are as follows:

In [98]:
# 🐍 slicing by explicit index / 将显式索引作为切片
data['2':'4']
data[2:4]

3    0.75
4     1.0
dtype: object

In [17]:
# 🐍 slicing by implicit integer index / 将隐式整数索引作为切片:左闭右开
data[0:2] 

1    0.25
2     0.5
dtype: object

In [99]:
# 🐍 masking / 掩码：条件判断 --> 读取条件为True的结果
data[(data != 1.0)]

a    0.25
2     0.5
3    0.75
y     new
e       e
dtype: object

In [100]:
# 🐍 fancy indexing / 花哨的索引: 指定索引
data[['e', 'y', '2']]

e      e
y    new
2    0.5
dtype: object

Among these, slicing may be the source of the most confusion.
Notice that when slicing with an explicit index (i.e., ``data['a':'c']``), the final index is *included* in the slice, while when slicing with an implicit index (i.e., ``data[0:2]``), the final index is *excluded* from the slice.

### 3. Indexers: loc, iloc, and ix / 索引器

These slicing and indexing conventions can be a source of confusion.
For example, if your ``Series`` has an explicit integer index, an indexing operation such as ``data[1]`` will use the explicit indices, while a slicing operation like ``data[1:3]`` will use the implicit Python-style index.

In [101]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data

1    a
3    b
5    c
dtype: object

In [21]:
# 🐍 explicit index when indexing / 取值操作: 显式索引
data[3]

'b'

In [22]:
# 🐍 implicit index when slicing / 切片操作：隐式索引
data[1:3] # 隐式索引范围[1,3)

3    b
5    c
dtype: object

Because of this potential confusion in the case of integer indexes, Pandas provides some special *indexer* attributes that explicitly expose certain indexing schemes.
These are not functional methods, but attributes that expose a particular slicing interface to the data in the ``Series``.

First, the ``loc`` attribute allows indexing and slicing that always references the explicit index:

🐍 第一种索引器是 loc 属性，表示取值和切片都是显式的

In [102]:
print(data)

data.loc[1]

1    a
3    b
5    c
dtype: object


'a'

In [24]:
data.loc[1:3]

1    a
3    b
dtype: object

The ``iloc`` attribute allows indexing and slicing that always references the implicit Python-style index:

🐍 第二种是 iloc 属性，表示取值和切片都是Python 形式的1 隐式索引

In [41]:
data.iloc[1]

'b'

In [26]:
data.iloc[1:3]

3    b
5    c
dtype: object

In [42]:
print(data)
print("\n")

print(data.iloc[1], data.loc[1])
print("\n")

print(data.iloc[1:3])
print(data.loc[1:3])

1    a
3    b
5    c
dtype: object


b a


3    b
5    c
dtype: object
1    a
3    b
dtype: object


A third indexing attribute, ``ix``, is a hybrid of the two, and for ``Series`` objects is equivalent to standard ``[]``-based indexing.
The purpose of the ``ix`` indexer will become more apparent in the context of ``DataFrame`` objects, which we will discuss in a moment.

🐍 第三种取值属性是 ix，它是前两种索引器的混合形式，在Series 对象中ix 等价于标准的[]（Python 列表）取值方式。

🐍 ix 索引器主要用于DataFrame 对象，后面将会介绍。

One guiding principle of Python code is that "explicit is better than implicit."
The explicit nature of ``loc`` and ``iloc`` make them very useful in maintaining clean and readable code; especially in the case of integer indexes, I recommend using these both to make code easier to read and understand, and to prevent subtle bugs due to the mixed indexing/slicing convention.

🐍 Python 代码的设计原则之一是“显式优于隐式”

## 3.2.2 Data Selection in DataFrame / DataFrame 数据选择方法

Recall that a ``DataFrame`` acts in many ways like a two-dimensional or structured array, and in other ways like a dictionary of ``Series`` structures sharing the same index.
These analogies can be helpful to keep in mind as we explore data selection within this structure.

🐍 DataFrame 在有些方面像二维或结构化数组

🐍 有些方面又像一个共享索引的若干Series 对象构成的字典

### 1. DataFrame as a dictionary  / 将 DataFrame 看作字典

The first analogy we will consider is the ``DataFrame`` as a dictionary of related ``Series`` objects.
Let's return to our example of areas and populations of states:

In [110]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area':area, 'pop':pop})
data

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


The individual ``Series`` that make up the columns of the ``DataFrame`` can be accessed via dictionary-style indexing of the column name:

In [105]:
# 🐍 对列名进行字典形式（dictionary-style）的取值获取数据

data['area']

type(data['area'])

pandas.core.series.Series

Equivalently, we can use attribute-style access with column names that are strings:

In [57]:
# 🐍 用属性形式（attribute-style）选择纯字符串列名的数据

data.area

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

This attribute-style column access actually accesses the exact same object as the dictionary-style access:

In [106]:
# 🐍 同一个对象进行属性形式与字典形式的列数据，结果是相同的

data.area

data.area is data['area']

True

Though this is a useful shorthand, keep in mind that it does not work for all cases!
For example, if the column names are not strings, or if the column names conflict with methods of the ``DataFrame``, this attribute-style access is not possible.
For example, the ``DataFrame`` has a ``pop()`` method, so ``data.pop`` will point to this rather than the ``"pop"`` column:

🐍 虽然属性形式的数据选择方法很方便，但是它并不是通用的

In [107]:
data.pop is data['pop']

False

In particular, you should avoid the temptation to try column assignment via attribute (i.e., use ``data['pop'] = z`` rather than ``data.pop = z``).

🐍 避免对用属性形式选择的列直接赋值（即可以用data['pop'] = z，但不要用data.pop = z）

Like with the ``Series`` objects discussed earlier, this dictionary-style syntax can also be used to modify the object, in this case adding a new column:

In [111]:
# data.pop = 'z'

# data['pop'] = 'z'
data

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


In [112]:
# 🐍 用字典形式的语法调整对象，增加一列

data['density'] = data['pop'] / data['area']
data

,area,pop,density
California,423967,38332521,90.413926
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


This shows a preview of the straightforward syntax of element-by-element arithmetic between ``Series`` objects; we'll dig into this further in [Operating on Data in Pandas](03.03-Operations-in-Pandas.ipynb).

### 2. DataFrame as two-dimensional array / 将 DataFrame 看作二维数组

As mentioned previously, we can also view the ``DataFrame`` as an enhanced two-dimensional array.
We can examine the raw underlying data array using the ``values`` attribute:

🐍 values 属性按行查看数组数据

In [70]:
data.values

array([[4.23967000e+05, 3.83325210e+07, 9.04139261e+01],
       [6.95662000e+05, 2.64481930e+07, 3.80187404e+01],
       [1.41297000e+05, 1.96511270e+07, 1.39076746e+02],
       [1.70312000e+05, 1.95528600e+07, 1.14806121e+02],
       [1.49995000e+05, 1.28821350e+07, 8.58837628e+01]])

With this picture in mind, many familiar array-like observations can be done on the ``DataFrame`` itself.
For example, we can transpose the full ``DataFrame`` to swap rows and columns:

🐍 对 DataFrame 进行行列转置：.T

In [71]:
data.T

,California,Texas,New York,Florida,Illinois
area,4.239670e+05,6.956620e+05,1.412970e+05,1.703120e+05,1.499950e+05
pop,3.833252e+07,2.644819e+07,1.965113e+07,1.955286e+07,1.288214e+07
density,9.041393e+01,3.801874e+01,1.390767e+02,1.148061e+02,8.588376e+01


When it comes to indexing of ``DataFrame`` objects, however, it is clear that the dictionary-style indexing of columns precludes our ability to simply treat it as a NumPy array.
In particular, passing a single index to an array accesses a row:

In [79]:
# 🐍 values方法无法赋值，它只是字典所有值的视图对象, 转换为列表后才可以
data.values[0] = 1

print(data.values)

data_list = data.values.tolist() #data_list = list(data.values)

data_list[0][0] = 1

data_list

[[4.23967000e+05 3.83325210e+07 9.04139261e+01]
 [6.95662000e+05 2.64481930e+07 3.80187404e+01]
 [1.41297000e+05 1.96511270e+07 1.39076746e+02]
 [1.70312000e+05 1.95528600e+07 1.14806121e+02]
 [1.49995000e+05 1.28821350e+07 8.58837628e+01]]


[[1, 38332521.0, 90.41392608386974],
 [695662.0, 26448193.0, 38.01874042279153],
 [141297.0, 19651127.0, 139.07674614464568],
 [170312.0, 19552860.0, 114.80612053173],
 [149995.0, 12882135.0, 85.88376279209307]]

and passing a single "index" to a ``DataFrame`` accesses a column:

In [38]:
data['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

Thus for array-style indexing, we need another convention.
Here Pandas again uses the ``loc``, ``iloc``, and ``ix`` indexers mentioned earlier.
Using the ``iloc`` indexer, we can index the underlying array as if it is a simple NumPy array (using the implicit Python-style index), but the ``DataFrame`` index and column labels are maintained in the result:

🐍 进行数组形式的取值时，我们就需要用另一种方法——前面介绍过的Pandas 索引器 loc、iloc 和 ix 

In [80]:
data.iloc[:3, :2]

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127


Similarly, using the ``loc`` indexer we can index the underlying data in an array-like style but using the explicit index and column names:

In [81]:
data.loc[:'Illinois', :'pop']

,area,pop
California,423967,38332521
Texas,695662,26448193
New York,141297,19651127
Florida,170312,19552860
Illinois,149995,12882135


The ``ix`` indexer allows a hybrid of these two approaches:

🐍 使用 ix 索引器可以实现一种混合效果 - 已停用

In [33]:
# 🐍 在较旧的版本中，Pandas的DataFrame确实有一个名为ix的方法，用于进行索引和切片操作。
# 🐍 然而，从Pandas版本0.20.0开始，ix方法已经被弃用，不再推荐使用，并且在最新的版本中已经被移除。

# data.ix[:3, :'pop']

Keep in mind that for integer indices, the ``ix`` indexer is subject to the same potential sources of confusion as discussed for integer-indexed ``Series`` objects.

Any of the familiar NumPy-style data access patterns can be used within these indexers.
For example, in the ``loc`` indexer we can combine masking and fancy indexing as in the following:

🐍 在 loc 索引器中结合使用掩码与花哨的索引方法

In [82]:
data.loc[data.density > 100, ['pop', 'density']]

,pop,density
New York,19651127,139.076746
Florida,19552860,114.806121


Any of these indexing conventions may also be used to set or modify values; this is done in the standard way that you might be accustomed to from working with NumPy:

In [84]:
# 🐍 调整数据-->赋值

data.iloc[0, 2] = 60
data

,area,pop,density
California,423967,38332521,60.000000
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


To build up your fluency in Pandas data manipulation, I suggest spending some time with a simple ``DataFrame`` and exploring the types of indexing, slicing, masking, and fancy indexing that are allowed by these various indexing approaches.

### 3. Additional indexing conventions / 其它取值方法

There are a couple extra indexing conventions that might seem at odds with the preceding discussion, but nevertheless can be very useful in practice.
First, while *indexing* refers to columns, *slicing* refers to rows:

🐍 如果对单个标签取值就选择列，而对多个标签用切片就选择行

In [85]:
data['Florida':'Illinois']

,area,pop,density
Florida,170312,19552860,114.806121
Illinois,149995,12882135,85.883763


Such slices can also refer to rows by number rather than by index:

In [86]:
# 🐍 切片对行取值，[1,3) 区间

data[1:3]

,area,pop,density
Texas,695662,26448193,38.018740
New York,141297,19651127,139.076746


Similarly, direct masking operations are also interpreted row-wise rather than column-wise:

🐍 掩码操作也可以直接对每一行进行过滤，而不需要使用 loc 索引器

In [87]:
data[data.density > 100]

,area,pop,density
New York,141297,19651127,139.076746
Florida,170312,19552860,114.806121


These two conventions are syntactically similar to those on a NumPy array, and while these may not precisely fit the mold of the Pandas conventions, they are nevertheless quite useful in practice.

<!--NAVIGATION-->
< [Introducing Pandas Objects](03.01-Introducing-Pandas-Objects.ipynb) | [Contents](Index.ipynb) | [Operating on Data in Pandas](03.03-Operations-in-Pandas.ipynb) >